# L13 · Meta-learners y efectos heterogéneos

**S13 · Lectura: Cap. 12 · 40–55 min**

## La pregunta

¿Dos personas con el mismo riesgo tienen el mismo efecto?

## Qué harás

Comparar formas de estimar CATE.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** CATE, PEHE y placebo sobre IHDP.

> **Idea clave:** El riesgo predicho dice quién puede tener un mal resultado; el CATE dice a quién le cambia actuar.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import Lasso, LogisticRegression
rng = setup()
N_REP = R(3, 20)
print("repeticiones por tamaño:", N_REP)

## 1. El mundo y el objetivo (§12.1)
Mundo del capítulo (Listado 12.1): $X\sim U(-1,1)^{10}$, propensity $e=\sigma(1+0{,}5X_0)$, $\mu_0=X_2+0{,}5X_3$ y efecto **heterogéneo** $\tau(x)=2+3X_0-X_1^2$.

Los cuatro meta-learners, en una línea cada uno:
- **S:** un solo modelo $\hat\mu(X,D)$; $\hat\tau=\hat\mu(x,1)-\hat\mu(x,0)$.
- **T:** dos modelos, uno por brazo: $\hat\tau=\hat\mu_1(x)-\hat\mu_0(x)$.
- **X:** imputa efectos individuales con el modelo del otro brazo, los regresa sobre $X$ y mezcla con $\hat e$.
- **R:** regresa los residuos $\tilde Y$ sobre $\tilde D$ con pesos $\tilde D^2$ (Robinson, Cap. 10 con $\tau$ variable).

In [ ]:
def make_dgp(n=2000, p=10, seed=0, a=1.0, b=0.5):
    r = np.random.default_rng(seed); X = r.uniform(-1, 1, (n, p)); e = 1 / (1 + np.exp(-a - b * X[:, 0]))
    D = (r.uniform(size=n) < e).astype(int); tau = 2 + 3 * X[:, 0] - X[:, 1] ** 2; mu0 = X[:, 2] + .5 * X[:, 3]
    Y = mu0 + D * tau + r.standard_normal(n) * .5
    return Y, D, X, tau
Y, D, X, tau_true = make_dgp(2000, seed=42)
Yt, Dt, Xt, tau_t = make_dgp(2000, seed=7)          # conjunto de PRUEBA con otra semilla: evaluamos fuera de muestra
ML = lambda: HistGradientBoostingRegressor(max_iter=100, learning_rate=.1, max_depth=4, random_state=0)
CL = lambda: HistGradientBoostingClassifier(max_iter=60, learning_rate=.1, max_depth=3, random_state=0)
print("tasa de tratamiento:", D.mean().round(3), "· τ medio verdadero:", tau_true.mean().round(3))

## 2. S y X ya hechos; T y R los completas tú
Aquí tienes `s_learner` y `x_learner`. 

### Completa tú
- `t_learner(Y, D, X, Xnew, make)`: dos modelos, uno por brazo.
- `r_learner(Y, D, X, Xnew, make_reg, make_clf, K=5)`: (1) con **cross-fitting** obtén $\hat m(X)=E[Y|X]$ y $\hat e(X)$; (2) residuos $\tilde Y=Y-\hat m$, $\tilde D=D-\hat e$; (3) ajusta un regresor de $\tilde Y/\tilde D$ sobre $X$ con `sample_weight` $=\tilde D^2$ y predice en `Xnew`.

In [ ]:
def s_learner(Y, D, X, Xnew, make):
    m = make().fit(np.column_stack([X, D]), Y)
    return m.predict(np.column_stack([Xnew, np.ones(len(Xnew))])) - m.predict(np.column_stack([Xnew, np.zeros(len(Xnew))]))

def x_learner(Y, D, X, Xnew, make):
    m1, m0 = make().fit(X[D == 1], Y[D == 1]), make().fit(X[D == 0], Y[D == 0])
    g1 = make().fit(X[D == 1], Y[D == 1] - m0.predict(X[D == 1]))
    g0 = make().fit(X[D == 0], m1.predict(X[D == 0]) - Y[D == 0])
    e = LogisticRegression(max_iter=500).fit(X, D).predict_proba(Xnew)[:, 1]
    return e * g0.predict(Xnew) + (1 - e) * g1.predict(Xnew)

In [ ]:
# ✏️ Completa tú — t_learner y r_learner
# Pista: T: dos .fit, uno con X[D==1] y otro con X[D==0]. R: el regresor final usa y = ry/rd y sample_weight = rd**2.
def t_learner(Y, D, X, Xnew, make):
    """Un modelo por brazo; devuelve el CATE estimado en Xnew."""
    raise NotImplementedError("completa t_learner")

def r_learner(Y, D, X, Xnew, make_reg, make_clf, K=5):
    """R-learner con cross-fitting (kfold(n, K, SEED)). Devuelve el CATE estimado en Xnew."""
    raise NotImplementedError("completa r_learner")

In [ ]:
def rmse(h, t): return float(np.sqrt(np.mean((h - t) ** 2)))
Xs, Ys, Ds = X[:600], Y[:600], D[:600]; Xn = Xt[:300]; tn = tau_t[:300]
h_t = t_learner(Ys, Ds, Xs, Xn, ML); h_r = r_learner(Ys, Ds, Xs, Xn, ML, CL)
lab_check("t_learner y r_learner (correlación con el τ verdadero y escala)", lambda: (np.corrcoef(h_t, tn)[0, 1] > .8, np.corrcoef(h_r, tn)[0, 1] > .7, abs(h_t.mean() - tn.mean()) < .6, abs(h_r.mean() - tn.mean()) < .8),
          [((), (True, True, True, True), 0)])

## 3. La tabla del capítulo: RMSE y sesgo (§12.3)
Se evalúa en el conjunto de **prueba** (el libro lo hace en la misma muestra; fuera de muestra es lo honesto). El τ verdadero solo existe porque simulamos.

## Sección de evaluación reservada

La edición pública termina antes de las celdas docentes que comparan con verdad simulada, efectos potenciales o resultados oráculo. Esa revisión se ejecuta aparte; aquí debes sostener la decisión con observables, incertidumbre y supuestos declarados.
